XGBoost

In [ ]:
# XGBoost
xgb = XGBClassifier(random_state=42)
xgb.fit(X_train_resampled, y_train_resampled)

# Predict probabilities on the test set
y_pred_proba = xgb.predict_proba(X_test_rfe)[:, 1]

# Evaluate ROC-AUC
roc_auc_xgb = roc_auc_score(y_test, y_pred_proba)
print(f'ROC-AUC: {roc_auc_xgb:.4f}')

In [ ]:
#Precision-Recall AUC
pr_auc_xgb = average_precision_score(y_test, y_pred_proba)
print(f'Precision-Recall AUC: {pr_auc_xgb:.4f}')
table.add_row(["XGBoost", f"{roc_auc_xgb:.4f}", f"{pr_auc_xgb:.4f}"])

In [ ]:
# Classification Report
y_pred = xgb.predict(X_test_rfe)
class_report = classification_report(y_test, y_pred) 
print(class_report)

Focal loss

In [ ]:
y_train_resampled = y_train_resampled.to_numpy().flatten()

In [ ]:
def score_eval_func(y_test, y_pred_prob):
    roc_auc = roc_auc_score(y_test, y_pred_prob[:, 1])
    pr_auc = average_precision_score(y_test, y_pred_prob[:, 1])
    return roc_auc, pr_auc


In [ ]:
#In FC the scaling factor decays to zero as confidence in the correct class increases
# down-weights the loss assigned to well-classified examples. easy examples are given less weight than misclassified examples.
# Instantiate imbalance-xgboost instance with focal loss
xgbooster_focal = imb_xgb(special_objective='focal')

# GridSearchCV for focal loss hyperparameter tuning
cvFocalBooster = GridSearchCV(xgbooster_focal, {'focal_gamma': [1.0, 1.5, 2.0, 2.5, 3.0]}, error_score='raise')

# Fit booster
cvFocalBooster.fit(X_train_resampled, y_train_resampled)
opt_focal_booster = cvFocalBooster.best_estimator_
opt_focal_parameter = cvFocalBooster.best_params_

# Fit the model on the entire dataset
opt_focal_booster.fit(X_train_resampled, y_train_resampled)

# Predict probabilities on the test set
# Evaluate ROC-AUC and Precision-Recall AUC
y_pred_prob_focal = opt_focal_booster.predict_two_class(X_test_rfe, y=None)
roc_auc_focal, pr_auc_focal = score_eval_func(y_test, y_pred_prob_focal)
print(f'ROC-AUC (Focal Loss): {roc_auc_focal:.4f}, Precision-Recall AUC: {pr_auc_focal:.4f}')
table.add_row(["Focal Loss", f"{roc_auc_focal:.4f}", f"{pr_auc_focal:.4f}"])

Weighted loss

In [ ]:
#The weights are used to assign a higher penalty to misclassifications of minority class. 
# Instantiate imbalance-xgboost instance with weighted loss
xgbooster_weight = imb_xgb(special_objective='weighted')

# GridSearchCV for weighted loss hyperparameter tuning
CV_weight_booster = GridSearchCV(xgbooster_weight, {"imbalance_alpha": [1.5, 2.0, 2.5, 3.0, 4.0]}, error_score='raise')

# Fit booster
CV_weight_booster.fit(X_train_resampled, y_train_resampled)
opt_weight_booster = CV_weight_booster.best_estimator_
opt_weight_parameter = CV_weight_booster.best_params_

# Fit the model on the entire dataset
opt_weight_booster.fit(X_train_resampled, y_train_resampled)

# Predict probabilities on the test set
# Evaluate ROC-AUC and Precision-Recall AUC
y_pred_prob_weight = opt_weight_booster.predict_two_class(X_test_rfe, y=None)
roc_auc_weight, pr_auc_weight = score_eval_func(y_test, y_pred_prob_weight)
print(f'ROC-AUC (Weighted Loss): {roc_auc_weight:.4f}, Precision-Recall AUC: {pr_auc_weight:.4f}')
table.add_row(["Weightd Loss", f"{roc_auc_weight:.4f}", f"{pr_auc_weight:.4f}"])

Grid Search

In [ ]:
# Define the parameter grid for Grid Search
param_grid = {
    'learning_rate': [0.01, 0.1],
    'max_depth': [7,9],
    'subsample': [1.0, 1.2],
    'scale_pos_weight': [5,7],
    'n_estimators': [200,300],
}

# Create the XGBoost classifier
xgb = XGBClassifier(random_state=42)

In [ ]:
# Base Model Hyperparameter Tuning
CV_base_booster = GridSearchCV(estimator=xgb, param_grid=param_grid, scoring='roc_auc') # precision - recall or f1
CV_base_booster.fit(X_train_resampled, y_train_resampled)
opt_base_booster = CV_base_booster.best_estimator_
print("Best Hyperparameters:", CV_base_booster.best_params_)

Base, Focal Loss and Weighted Loss model after Hyperparameter Tuning

In [ ]:
# function to evaluate results
def score_eval_func(y_test, y_pred_prob):
    # Calculate ROC-AUC and Precision-Recall AUC
    roc_auc = roc_auc_score(y_test, y_pred_prob)
    precision, recall, _ = precision_recall_curve(y_test, y_pred_prob)
    pr_auc = auc(recall, precision)
    
    return roc_auc, pr_auc

# Predict probabilities on the test set for the base model
y_pred_prob_base = opt_base_booster.predict_proba(X_test_rfe)[:, 1] 
 

# Evaluate ROC-AUC and Precision-Recall AUC 
roc_auc_gs, pr_auc_gs = score_eval_func(y_test, y_pred_prob_base)
print(f'Base Model - ROC-AUC: {roc_auc_gs:.4f}, Precision-Recall AUC: {pr_auc_gs:.4f}')
table.add_row(["Grid Search_Base Model", f"{roc_auc_gs:.4f}", f"{pr_auc_gs:.4f}"])

In [ ]:
# Predict probabilities on the validation set using predict_two_class
y_pred_prob_focal = opt_focal_booster.predict_two_class(X_test_rfe, y=None)

# Extract the probability scores for the positive class
y_pred_prob_positive_class = y_pred_prob_focal[:, 1]

# Evaluate ROC-AUC and Precision-Recall AUC 
roc_auc_gs_focal, pr_auc_gs_focal = score_eval_func(y_test, y_pred_prob_positive_class)
print(f'Focal Loss Model - ROC-AUC: {roc_auc_gs_focal:.4f}, Precision-Recall AUC: {pr_auc_gs_focal:.4f}')
table.add_row(["Grid Search_Focal Loss", f"{roc_auc_gs_focal:.4f}", f"{pr_auc_gs_focal:.4f}"])

In [ ]:
# Predict probabilities on the validation set using predict_two_class
y_pred_prob_weight = opt_weight_booster.predict_two_class(X_test_rfe, y=None)

# Extract the probability scores for the positive class  
y_pred_prob_positive_class_weight = y_pred_prob_weight[:, 1]

# Evaluate ROC-AUC and Precision-Recall AUC for the weighted loss model
roc_auc_gs_weight, pr_auc_gs_weight = score_eval_func(y_test, y_pred_prob_positive_class_weight)
print(f'Weighted Loss Model - ROC-AUC: {roc_auc_gs_weight:.4f}, Precision-Recall AUC: {pr_auc_gs_weight:.4f}')
table.add_row(["Grid Search_Weighted Loss", f"{roc_auc_gs_weight:.4f}", f"{pr_auc_gs_weight:.4f}"])

In [ ]:
print(table)

Winning : XGBoost with Grid Search Base Model